# 01 - Load and profile

Loads the six raw CSVs and runs a first pass of checks: row counts vs. the brief, column dtypes, date ranges, and the value distributions we'll lean on later (status, delivery_status, segment, carrier, category).

Nothing is written out yet. Cleaning and feature engineering happen in `02_clean_features.ipynb`.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

DATA_DIR = Path('..').resolve().parent
DATA_DIR

WindowsPath('L:/OJ Commerece assessment')

In [2]:
customers = pd.read_csv(DATA_DIR / 'customers.csv')
sellers   = pd.read_csv(DATA_DIR / 'sellers.csv')
products  = pd.read_csv(DATA_DIR / 'products.csv')
orders    = pd.read_csv(DATA_DIR / 'orders.csv')
items     = pd.read_csv(DATA_DIR / 'order_items.csv')
shipments = pd.read_csv(DATA_DIR / 'shipments.csv')

tables = {
    'customers': customers,
    'sellers':   sellers,
    'products':  products,
    'orders':    orders,
    'items':     items,
    'shipments': shipments,
}

{name: df.shape for name, df in tables.items()}

{'customers': (25000, 5),
 'sellers': (400, 4),
 'products': (3000, 4),
 'orders': (100000, 7),
 'items': (169929, 8),
 'shipments': (91994, 9)}

Brief expects roughly: 25k customers, 400 sellers, 3k products, 100k orders, 170k items, 92k shipments. Check the numbers above match before moving on.

In [3]:
for name, df in tables.items():
    print(f'--- {name} ---')
    print(df.dtypes)
    print()

--- customers ---
customer_id    object
signup_date    object
city           object
state          object
segment        object
dtype: object

--- sellers ---
seller_id        object
seller_name      object
primary_city     object
rating          float64
dtype: object

--- products ---
product_id      object
category        object
subcategory     object
base_price     float64
dtype: object

--- orders ---
order_id                     object
customer_id                  object
created_at                   object
status                       object
payment_method               object
promised_delivery_date       object
is_fast_delivery_eligible      bool
dtype: object

--- items ---
order_item_id        object
order_id             object
product_id           object
seller_id            object
quantity              int64
unit_price          float64
discount_pct        float64
platform_fee_pct    float64
dtype: object

--- shipments ---
shipment_id         object
order_id            object

In [4]:
for name, df in tables.items():
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        print(f'{name}: nulls present')
        print(nulls)
    else:
        print(f'{name}: no nulls')

customers: no nulls
sellers: no nulls
products: no nulls
orders: no nulls
items: no nulls
shipments: nulls present
delivered_at    5473
dtype: int64


In [5]:
orders['created_at'] = pd.to_datetime(orders['created_at'])
orders['promised_delivery_date'] = pd.to_datetime(orders['promised_delivery_date'])
shipments['shipped_at']   = pd.to_datetime(shipments['shipped_at'])
shipments['delivered_at'] = pd.to_datetime(shipments['delivered_at'])
customers['signup_date']  = pd.to_datetime(customers['signup_date'])

print('orders.created_at      ', orders['created_at'].min(),    '->', orders['created_at'].max())
print('orders.promised        ', orders['promised_delivery_date'].min(), '->', orders['promised_delivery_date'].max())
print('shipments.shipped_at   ', shipments['shipped_at'].min(),    '->', shipments['shipped_at'].max())
print('shipments.delivered_at ', shipments['delivered_at'].min(),  '->', shipments['delivered_at'].max())
print('customers.signup_date  ', customers['signup_date'].min(),   '->', customers['signup_date'].max())

orders.created_at       2024-07-01 00:00:00 -> 2025-12-30 00:00:00
orders.promised         2024-07-03 00:00:00 -> 2026-01-04 00:00:00
shipments.shipped_at    2024-07-01 04:48:15 -> 2025-12-30 16:47:46
shipments.delivered_at  2024-07-01 16:07:59 -> 2026-01-04 16:05:11
customers.signup_date   2022-07-03 00:00:00 -> 2024-07-01 00:00:00


In [6]:
print('orders.status')
print(orders['status'].value_counts(dropna=False))
print()
print('orders.payment_method')
print(orders['payment_method'].value_counts(dropna=False))
print()
print('orders.is_fast_delivery_eligible')
print(orders['is_fast_delivery_eligible'].value_counts(dropna=False))

orders.status
Delivered    82069
Cancelled     8006
Shipped       5043
Returned      4882
Name: status, dtype: int64

orders.payment_method
UPI           35024
COD           24888
Card          20127
NetBanking    11997
Wallet         7964
Name: payment_method, dtype: int64

orders.is_fast_delivery_eligible
True     65090
False    34910
Name: is_fast_delivery_eligible, dtype: int64


In [7]:
print('shipments.carrier')
print(shipments['carrier'].value_counts(dropna=False))
print()
print('shipments.delivery_status')
print(shipments['delivery_status'].value_counts(dropna=False))

shipments.carrier
Delhivery    29539
BlueDart     21906
Ekart        21840
InHouse      18709
Name: carrier, dtype: int64

shipments.delivery_status
OnTime       63682
Late_1_2d    20388
InTransit     5043
Late_3_5d     2424
Lost           430
Late_5p         27
Name: delivery_status, dtype: int64


In [8]:
print('customers.segment')
print(customers['segment'].value_counts(dropna=False))
print()
print('customers.city')
print(customers['city'].value_counts(dropna=False))
print()
print('products.category')
print(products['category'].value_counts(dropna=False))

customers.segment
Budget     10170
Value       9921
Premium     4909
Name: segment, dtype: int64

customers.city
Mumbai        4455
Delhi         3965
Bangalore     3518
Hyderabad     2538
Chennai       2259
Pune          1997
Kolkata       1739
Ahmedabad     1234
Jaipur        1029
Lucknow        992
Chandigarh     764
Kochi          510
Name: city, dtype: int64

products.category
Electronics       835
Fashion           778
Grocery           559
Home & Kitchen    501
Books             327
Name: category, dtype: int64


## Quick referential integrity

Spot-check the foreign keys before we trust joins downstream.

In [9]:
checks = {
    'orders without customer':    (~orders['customer_id'].isin(customers['customer_id'])).sum(),
    'items without order':        (~items['order_id'].isin(orders['order_id'])).sum(),
    'items without product':      (~items['product_id'].isin(products['product_id'])).sum(),
    'items without seller':       (~items['seller_id'].isin(sellers['seller_id'])).sum(),
    'shipments without order':    (~shipments['order_id'].isin(orders['order_id'])).sum(),
    'orders missing a shipment':  (~orders['order_id'].isin(shipments['order_id'])).sum(),
    'orders missing items':       (~orders['order_id'].isin(items['order_id'])).sum(),
}
checks

{'orders without customer': 0,
 'items without order': 0,
 'items without product': 0,
 'items without seller': 0,
 'shipments without order': 0,
 'orders missing a shipment': 8006,
 'orders missing items': 0}

Note the `orders missing a shipment` count. Cancelled orders likely never ship, so we expect this to be non-zero. We'll cross-tab it against `orders.status` next.

In [10]:
shipped_ids = set(shipments['order_id'])
orders_no_ship = orders[~orders['order_id'].isin(shipped_ids)]
orders_no_ship['status'].value_counts(dropna=False)

Cancelled    8006
Name: status, dtype: int64

Sanity on time ordering: `created_at <= shipped_at <= delivered_at` should hold for delivered shipments.

In [11]:
ts = shipments.merge(orders[['order_id', 'created_at']], on='order_id', how='left')
delivered = ts[ts['delivered_at'].notna()].copy()

bad_ship_before_create = (delivered['shipped_at']   < delivered['created_at']).sum()
bad_deliver_before_ship = (delivered['delivered_at'] < delivered['shipped_at']).sum()

print('shipped before created  :', bad_ship_before_create)
print('delivered before shipped:', bad_deliver_before_ship)
print('delivered rows total    :', len(delivered))

shipped before created  : 0
delivered before shipped: 0
delivered rows total    : 86521
